# Extracció de Pesos CAV i Interpretabilitat Basada en Conceptes sobre TNet

---

## 1. Introducció i Motivació

### 1.1 Context: Interpretabilitat en Xarxes Neuronals Profundes

Les xarxes neuronals convolucionals (CNN) han demostrat un rendiment excepcional en tasques de classificació d'imatges, però el seu funcionament intern roman opac o completament oscur, això és coneix com a problema de **caixa negra** (*black box*). Comprendre *per què* un model pren una decisió concreta és essencial tant per a la validació científica com per a l'aplicació en entorns reals.

Els mètodes clàssics d'atribució de característiques (p. ex., **saliency maps**, **Grad-CAM**) operen a nivell de píxel, generant mapes de calor que indiquen quines regions de la imatge influeixen la predicció. Tot i ser útils, aquests mètodes no responen directament a la pregunta:

> El model ha après a reconèixer conceptes d'alt nivell com "cercle", "quadrat" o "creu"? I si és així, a quines capes internes es codifiquen?

### 1.2 TCAV: Testing with Concept Activation Vectors

**TCAV** (Kim et al., 2018) proporciona una metodologia per respondre aquesta pregunta. La idea central és:

1. Definir **conceptes** mitjançant conjunts d'imatges representatives (p. ex., imatges que contenen cercles, creus, quadrats, etc.).
2. Per a cada capa interna $\ell$ del model, recollir les **activacions intermèdies** quan es processen imatges del concepte i imatges aleatòries (control).
3. Entrenar un **classificador lineal** (regressió logística) per separar les activacions del concepte de les aleatòries. El vector de pesos d'aquest classificador defineix el **Vector d'Activació del Concepte (CAV)**:

$$\mathbf{v}_c^\ell \in \mathbb{R}^D$$

on $D$ és la dimensionalitat de l'espai d'activacions a la capa $\ell$.

4. Calcular la **derivada direccional** de la sortida del model en la direcció del CAV per quantificar quant influeix el concepte en la predicció.

### 1.3 Objectius d'aquest Notebook

Aquest notebook implementa un recorregut complet d'interpretabilitat basada en conceptes sobre un model personalitzat (**TNet**), amb tres contribucions principals:

1. **Extracció explícita dels pesos CAV** per a cada parell concepte-capa, amb anàlisi estadística de les magnituds.
2. **Projecció espacial dels pesos CAV** sobre mapes d'activació per generar **mapes de calor de localització de conceptes**, definir un model que no depèn del gradient de sortida.
3. **Diagnòstic d'un mode de fallada** de TCAV sota el que consider com a **saturació de la sigmoide**, i demostració que la projecció CAV el circumventa.

### 1.4 Referències

- Kim, B. et al. (2018). *Interpretability Beyond Feature Attribution: Quantitative Testing with Concept Activation Vectors (TCAV)*. ICML 2018.
- Kokhlikyan, N. et al. (2020). *Captum: A unified and generic model interpretability library for PyTorch*.
- Visual-Tcav 

---


# 2. Configuració de l'Entron i Importacions

In [1]:
import sys, os
os.chdir("/Users/dylancanning/Documents/TFG/Tcav_Captum/Testing-with-Concept-Activation-Vectors")

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from captum.concept import TCAV, Concept
from captum.concept._utils.data_iterator import dataset_to_dataloader, CustomIterableDataset

from model.model_sq import TNet

print("Importacions completades correctament.")
print(f"Directori de treball: {os.getcwd()}")

Importacions completades correctament.
Directori de treball: /Users/dylancanning/Documents/TFG/Tcav_Captum/Testing-with-Concept-Activation-Vectors


/opt/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## 3. Model Objectiu: Arquitectura TNet

### 3.1 Descripció de l'Arquitectura

**TNet** és una xarxa neuronal convolucional de 5 blocs convolucionals seguida de 3 capes fully connected, dissenyada per a **classificació binària** sobre imatges en escala de grisos de $128 \times 128$ píxels.

L'arquitectura segueix un patró típic on el nombre de canals augmenta progressivament mentre les dimensions espacials es redueixen per un factor de 2 a cada bloc (via `MaxPool2d`):

| Bloc | Capes | Forma de Sortida |
|------|-------|------------------|
| Conv 1 | `Conv2d(1, 25, 3×3, same)` → `ReLU` → `MaxPool2d(2×2)` | $[25, 64, 64]$ |
| Conv 2 | `Conv2d(25, 35, 3×3, same)` → `ReLU` → `MaxPool2d(2×2)` | $[35, 32, 32]$ |
| Conv 3 | `Conv2d(35, 50, 3×3, same)` → `ReLU` → `BatchNorm2d` → `MaxPool2d(2×2)` | $[50, 16, 16]$ |
| Conv 4 | `Conv2d(50, 75, 3×3, same)` → `ReLU` → `BatchNorm2d` → `MaxPool2d(2×2)` | $[75, 8, 8]$ |
| Conv 5 | `Conv2d(75, 125, 3×3, same)` → `ReLU` → `BatchNorm2d` → `MaxPool2d(2×2)` | $[125, 4, 4]$ |
| Flatten | — | $[2000]$ |
| FC 1 | `Linear(2000, 500)` → `Dropout(0.2)` → `ReLU` | $[500]$ |
| FC 2 | `Linear(500, 250)` → `Dropout(0.2)` → `ReLU` | $[250]$ |
| FC 3 | `Linear(250, 50)` → `Dropout(0.2)` → `ReLU` | $[50]$ |
| Sortida | `Linear(50, 1)` → `Sigmoid` | $[1]$ |

### 3.2 Funció de la Sigmoide a la Sortida

El model utilitza una funció **sigmoide** a la capa de sortida:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

que mapeja el logit $z \in \mathbb{R}$ a l'interval $(0, 1)$. La interpretació de la predicció és:

- $\hat{y} \approx 1.0$: La imatge **conté cercles** (classe "cercle present").
- $\hat{y} \approx 0.0$: La imatge és **dominant en quadrats sense cercles** (etiqueta $= 0$ si i només si `cercles == 0 AND quadrats > 0 AND quadrats >= creus`).

**Nota important per a TCAV:** La derivada de la sigmoide és $\sigma'(z) = \sigma(z)(1 - \sigma(z))$, que tendeix a zero quan $|z|$ és gran. Això tindrà conseqüències directes sobre les puntuacions TCAV, com veurem a la Secció 8.

### 3.3 Càrrega del Model

El model es posa en mode d'avaluació (`model.eval()`), que:
- Desactiva el **Dropout** (totes les neurones actives).
- Fa servir les estadístiques acumulades (*running mean/var*) del **BatchNorm** en comptes de les estadístiques del batch actual.

In [2]:
# =============================================
# Càrrega del model TNet pre-entrenat
# =============================================
PESOS = "weights/only_sq.pt"
DEVICE = torch.device("cpu")

obj = torch.load(PESOS, map_location=DEVICE, weights_only=True)
in_channels = obj["conv1.weight"].shape[1]
num_classes = obj["fc4.weight"].shape[0]

model = TNet(numChannels=in_channels, classes=num_classes, size_img=128)
model.to(DEVICE).eval()
model.load_state_dict(obj)

print(f"Model carregat: TNet(in_channels={in_channels}, classes={num_classes}) a {DEVICE}")
print(model)

Model carregat: TNet(in_channels=1, classes=1) a cpu
TNet(
  (conv1): Conv2d(1, 25, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (relu1): ReLU()
  (bn1): BatchNorm2d(25, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (maxpool1): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(25, 35, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (relu2): ReLU()
  (bn2): BatchNorm2d(35, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (maxpool2): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(35, 50, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (relu3): ReLU()
  (bn3): BatchNorm2d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (maxpool3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv4): Conv2d(50, 75, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (relu4): ReLU()
  (bn4

---

## 4. Preprocessament d'Imatges i Construcció de Conceptes

### 4.1 Transformacions

**Raonament:** Cada imatge d'entrada ha de passar per una seqüència de transformacions per adaptar-la al format que espera FlexNet:

1. **`Grayscale()`** — Converteix la imatge a un sol canal (escala de grisos). TNet espera `in_channels=1`.
2. **`Resize((128, 128))`** — Redimensiona a la resolució de l'arquitectura.
3. **`ToTensor()`** — Converteix la imatge PIL a un tensor `float32` normalitzat a $[0, 1]$.

Formalment, per a una imatge d'entrada $\mathbf{I} \in \{0, \ldots, 255\}^{H \times W}$, la transformació produeix:

$$\mathbf{X} = \frac{\text{Resize}(\text{Grayscale}(\mathbf{I}))}{255} \in [0, 1]^{1 \times 128 \times 128}$$

### 4.2 Funció de Construcció de Conceptes

**Raonament:** La biblioteca Captum requereix que cada concepte es representi com un objecte `Concept` amb un iterador de dades. Definim una funció `assemble_concept` que:

1. Construeix el camí al directori del concepte (p. ex., `data/concepts/circle_full/`).
2. Crea un `CustomIterableDataset` que carrega les imatges de forma *lazy* (sota demanda).
3. L'embolcalla en un `DataLoader` mitjançant `dataset_to_dataloader`.
4. Retorna un objecte `Concept` amb un ID únic, un nom i l'iterador de dades.


In [3]:
# =============================================
# Transformacions i conceptes
# =============================================
from torchvision import transforms

transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

def get_tensor_from_filename(filename):
    img = Image.open(filename).convert("L")
    return transform(img).to(DEVICE).float()

def assemble_concept(name, id, concepts_path="data/concepts"):
    concept_path = os.path.join(concepts_path, name) + "/"
    dataset = CustomIterableDataset(get_tensor_from_filename, concept_path)
    concept_iter = dataset_to_dataloader(dataset)
    return Concept(id=id, name=name, data_iter=concept_iter)

---

## 5. Assemblatge dels Conceptes Experimentals

### 5.1 Conceptes Geomètrics

**Raonament:** Definim tres conceptes d'alt nivell corresponents a les formes geomètriques presents al dataset AIXI:

| Concepte | ID | Descripció | Directori |
|----------|----|------------|----------|
| `circle_full` | 0 | Imatges que contenen cercles | `data/concepts/circle_full/` |
| `cross_full` | 1 | Imatges que contenen creus | `data/concepts/cross_full/` |
| `square_full` | 2 | Imatges que contenen quadrats | `data/concepts/square_full/` |

### 5.2 Conjunts Aleatoris per a control

Per a TCAV, els pools aleatoris serveixen com a classe negativa quan s'entrena el classificador lineal (CAV). La seva funció és proporcionar una línia base estadísticament neutra:

> Si la direcció CAV d'un concepte no és més informativa que una direcció que separa imatges aleatòries, llavors el concepte no està codificat de manera significativa a aquella capa.

| Pool | ID |
|------|----|
| `random_pool` | 100 |
| `random_pool_2` | 101 |
| `random_pool_3` | 102 |
| `random_pool_4` | 103 |

In [4]:
# =============================================
# Creació dels conceptes geomètrics i pools aleatoris
# =============================================
concepts_path = "./data/concepts"

# Conceptes de forma
circle_full = assemble_concept("circle_full", 0, concepts_path=concepts_path)
cross_full  = assemble_concept("cross_full",  1, concepts_path=concepts_path)
square_full = assemble_concept("square_full", 2, concepts_path=concepts_path)

# Pools aleatoris (control)
random_pool   = assemble_concept("random_pool",   100, concepts_path=concepts_path)
random_pool_2 = assemble_concept("random_pool_2", 101, concepts_path=concepts_path)
random_pool_3 = assemble_concept("random_pool_3", 102, concepts_path=concepts_path)
random_pool_4 = assemble_concept("random_pool_4", 103, concepts_path=concepts_path)

CONCEPTS = {
    "circle_full": circle_full,
    "cross_full":  cross_full,
    "square_full": square_full,
}

print(f"Conceptes assemblats: {list(CONCEPTS.keys())}")
print(f"Pools aleatoris: random_pool, random_pool_2, random_pool_3, random_pool_4")

Conceptes assemblats: ['circle_full', 'cross_full', 'square_full']
Pools aleatoris: random_pool, random_pool_2, random_pool_3, random_pool_4


---

## 6. Fonaments Teòrics dels Vectors d'Activació de Conceptes (CAV)

### 6.1 Intuïció Geomètrica

Considerem una capa interna $\ell$ d'una xarxa neuronal. Quan una imatge $\mathbf{X}$ passa per la xarxa, la capa $\ell$ produeix un tensor d'activacions $\mathbf{A}^\ell(\mathbf{X})$. Per a una capa convolucional amb $C$ canals i dimensions espacials $H \times W$:

$$\mathbf{A}^\ell(\mathbf{X}) \in \mathbb{R}^{C \times H \times W}$$

Per fer feina amb classificadors lineals, aplanem (*nitoriflatten*) aquest tensor a un vector:

$$\mathbf{a}^\ell(\mathbf{X}) = \text{flatten}(\mathbf{A}^\ell(\mathbf{X})) \in \mathbb{R}^D, \quad D = C \cdot H \cdot W$$

### 6.2 Entrenament del Classificador Lineal

Sigui $\mathcal{P}_c = \{\mathbf{X}_1^c, \ldots, \mathbf{X}_n^c\}$ el conjunt d'imatges del concepte $c$ i $\mathcal{P}_r = \{\mathbf{X}_1^r, \ldots, \mathbf{X}_m^r\}$ el pool aleatori. Construïm un dataset d'entrenament:

$$\mathcal{D} = \{(\mathbf{a}^\ell(\mathbf{X}_i^c), 1)\}_{i=1}^n \cup \{(\mathbf{a}^\ell(\mathbf{X}_j^r), 0)\}_{j=1}^m$$

i entrenem un classificador de **regressió logística** que aprèn una matriu de pesos $\mathbf{W} \in \mathbb{R}^{2 \times D}$:

$$P(y = 1 \mid \mathbf{a}) = \sigma(\mathbf{w}_1^\top \mathbf{a} + b_1)$$

on $\mathbf{w}_1$ és la primera fila de $\mathbf{W}$ (pesos per a la classe "concepte"). Aquesta fila defineix el **CAV**:

$$\boxed{\mathbf{v}_c^\ell = \mathbf{w}_1 \in \mathbb{R}^D}$$

El CAV és la **direcció en l'espai d'activacions** que separa maximalment les activacions del concepte de les aleatòries.

### 6.3 Selecció de Capes i Conjunts Experimentals

**Raonament sobre la selecció de capes:** Sondem les capes `conv3` a `fc3`, excloent `conv1` i `conv2`. La justificació és:

- **Capes primerenques** (`conv1`, `conv2`): Codifiquen característiques de baix nivell (vores, textures) que són compartides entre conceptes i, per tant, poc discriminatives.
- **Capes convolucionals mitjanes-tardanes** (`conv3`, `conv4`, `conv5`): Preserven estructura espacial i codifiquen patrons de nivell superior (fragments de formes).
- **Capes fully connected** (`fc1`, `fc2`, `fc3`): L'estructura espacial s'ha col·lapsat, però la informació s'ha comprimit en representacions compactes i potencialment més específiques del concepte.

**Conjunts experimentals:** Cada concepte es compara amb el pool aleatori principal:

$$\text{circle\_full vs random\_pool}, \quad \text{cross\_full vs random\_pool}, \quad \text{square\_full vs random\_pool}$$